# Incremental value of FTS data: Model C (LightGBM) on OOT

Compares the calibrated Model C PD (`prob_C_cal`) of the main v1.0.1 model (**with FTS**, challenger) and the v1.0.1-nofts variant (**without FTS**, baseline) on the same OOT applications: decile analysis, AUC / Gini / KS, swap-set analysis at equal approval rates, and a secondary view at equal raw PD cut-offs. Report only; outputs go to `v1.0.1_nofts/reports/`.

**Population**: OOT applications in both files with `sample_status` in GOOD / BAD (`config.labelled_statuses`), bad flag `target = 1`. EXCLUDED / EXCLUDED_PERF rows also carry a 0/1 target but are outside the labelled population (spec 2.3), so they are not used.

In [0]:
%pip install openpyxl --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, json, time, shutil, tempfile, importlib
from datetime import datetime
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)

# Baseline = this (variant) run's artifacts_root, challenger = main v1.0.1 artifacts_root
REPORTS   = os.path.join(common.artifacts_path(cfg), "reports")           # outputs go here only
BASE_CSV  = os.path.join(REPORTS, "oot_predictions.csv")                    # without FTS
CHAL_CSV  = os.path.join(common.main_artifacts_path(cfg), "reports", "oot_predictions.csv")   # with FTS
assert BASE_CSV == "/Volumes/mis_automation/datascience_pbg/pbg_cc_ascore/output/v1.0.1_nofts/reports/oot_predictions.csv", BASE_CSV
assert CHAL_CSV == "/Volumes/mis_automation/datascience_pbg/pbg_cc_ascore/output/v1.0.1/reports/oot_predictions.csv", CHAL_CSV

KEY, SCORE, TARGET, STATUS = "CREDITCARDNO", "prob_C_cal", "target", cfg["columns"]["status"]
LABELLED = cfg["labelled_statuses"]                   # GOOD, BAD
BAD_VALUE = 1
# Direction: HIGHER score = HIGHER risk (score is a PD). Approve the LOWEST scores; decile 1 = HIGHEST scores.
HIGHER_IS_RISKIER = True
NAMES = {"base": "Without FTS", "chal": "With FTS"}
APPROVAL_RATES = [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
RAW_CUTOFFS = [0.03, 0.05, 0.075, 0.10, 0.15, 0.20, 0.30]   # approve if PD < cutoff (secondary view)
OUT_PREFIX = "05c_fts_comparison_"

_TMP = tempfile.mkdtemp(prefix="s05c_")
def save_csv(df, name):
    tmp = os.path.join(_TMP, name)
    df.to_csv(tmp, index=False)
    shutil.copyfile(tmp, os.path.join(REPORTS, name))
    print(f"Saved {os.path.join(REPORTS, name)}")
def save_fig(fig, name):
    tmp = os.path.join(_TMP, name)
    fig.savefig(tmp, dpi=130, bbox_inches="tight")
    shutil.copyfile(tmp, os.path.join(REPORTS, name))
    print(f"Saved {os.path.join(REPORTS, name)}")

GIT_COMMIT, CONFIG_HASH = common.git_commit(PROJECT_ROOT), common.config_hash(cfg)
_t0 = time.time()
print(f"Baseline (without FTS): {BASE_CSV}\nChallenger (with FTS) : {CHAL_CSV}\nOutputs: {REPORTS}")

In [0]:
cols = [KEY, STATUS, TARGET, SCORE]
base = pd.read_csv(BASE_CSV, usecols=cols, dtype={KEY: str})
chal = pd.read_csv(CHAL_CSV, usecols=cols, dtype={KEY: str})
for n, d in (("base", base), ("chal", chal)):
    d[KEY] = d[KEY].str.strip().str.replace(r"\.0$", "", regex=True)
    assert d[KEY].is_unique, f"{NAMES[n]}: {KEY} not unique"

j = base.merge(chal, on=KEY, how="inner", suffixes=("_base", "_chal"), validate="one_to_one")
TARGET_MATCH = bool((j[f"{TARGET}_base"] == j[f"{TARGET}_chal"]).all())
STATUS_MATCH = bool((j[f"{STATUS}_base"] == j[f"{STATUS}_chal"]).all())
assert TARGET_MATCH and STATUS_MATCH, "bad flag / sample_status differ between the two files"

df = (j[j[f"{STATUS}_base"].isin(LABELLED)]
      .rename(columns={f"{TARGET}_base": "target", f"{STATUS}_base": "sample_status",
                       f"{SCORE}_base": "score_base", f"{SCORE}_chal": "score_chal"})
      [[KEY, "sample_status", "target", "score_base", "score_chal"]].reset_index(drop=True))
assert set(df["target"].unique()) <= {0, 1} and ((df["target"] == 1) == (df["sample_status"] == "BAD")).all()
df["bad"] = (df["target"] == BAD_VALUE).astype(int)
N, NB = len(df), int(df["bad"].sum())

COUNTS = pd.DataFrame([
    {"step": "rows without FTS file", "n": len(base)}, {"step": "rows with FTS file", "n": len(chal)},
    {"step": "in both (joined on CREDITCARDNO)", "n": len(j)},
    {"step": "only in without-FTS file", "n": len(set(base[KEY]) - set(chal[KEY]))},
    {"step": "only in with-FTS file", "n": len(set(chal[KEY]) - set(base[KEY]))},
    {"step": "GOOD+BAD (analysis population)", "n": N}, {"step": "bads", "n": NB}, {"step": "goods", "n": N - NB}])
display(COUNTS)
print(f"Bad flag identical in both files: {TARGET_MATCH} | sample_status identical: {STATUS_MATCH}")
print(f"Overall bad rate: {NB / N:.2%} ({NB:,} bads / {N:,})")

In [0]:
def risk(s):
    """Score oriented so that larger = riskier (explicit direction)."""
    return s if HIGHER_IS_RISKIER else -s

def decile_table(y, s, label):
    r = risk(pd.Series(s)).rank(method="first", ascending=False)        # 1 = riskiest application
    d = pd.qcut(r, 10, labels=range(1, 11)).astype(int)                 # 10 equal-count deciles, 1 = highest risk
    t = (pd.DataFrame({"decile": d, "y": y, "s": s}).groupby("decile")
         .agg(count=("y", "size"), bads=("y", "sum"), score_min=("s", "min"), score_max=("s", "max")).reset_index())
    t["goods"] = t["count"] - t["bads"]
    t["bad_rate"] = t["bads"] / t["count"]
    t["cum_bad_capture"] = t["bads"].cumsum() / t["bads"].sum()
    t["cum_good_share"] = t["goods"].cumsum() / t["goods"].sum()
    t["ks"] = (t["cum_bad_capture"] - t["cum_good_share"]).abs()
    t.insert(0, "model", label)
    return t[["model", "decile", "count", "bads", "goods", "bad_rate", "cum_bad_capture", "cum_good_share", "ks",
              "score_min", "score_max"]]

y = df["bad"].to_numpy()
DECILES = pd.concat([decile_table(y, df[f"score_{m}"].to_numpy(), NAMES[m]) for m in ("base", "chal")], ignore_index=True)

perf = []
for m in ("base", "chal"):
    s = risk(df[f"score_{m}"]).to_numpy()
    auc = roc_auc_score(y, s)
    perf.append({"model": NAMES[m], "n": N, "bads": NB, "auc": auc, "gini": 2 * auc - 1, "ks": common.ks_stat(y, s),
                 "ks_decile_max": DECILES.loc[DECILES.model == NAMES[m], "ks"].max(),
                 "top10_bad_capture": DECILES.loc[(DECILES.model == NAMES[m]) & (DECILES.decile == 1), "cum_bad_capture"].iloc[0],
                 "top20_bad_capture": DECILES.loc[(DECILES.model == NAMES[m]) & (DECILES.decile == 2), "cum_bad_capture"].iloc[0]})
PERF = pd.DataFrame(perf)

# Paired bootstrap of the difference (with FTS minus without FTS), same resamples for both models
PAIRED = common.paired_bootstrap(y, risk(df["score_chal"]).to_numpy(), risk(df["score_base"]).to_numpy(), cfg)
PAIRED.insert(0, "pair", "with FTS - without FTS")

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
for m in NAMES.values():
    print(f"\nDecile table: {m} (decile 1 = highest PD)")
    display(DECILES[DECILES.model == m])
print("\nOverall performance"); display(PERF)
print("\nPaired bootstrap, with FTS minus without FTS"); display(PAIRED)

In [0]:
def approve_at_rate(s, rate):
    """Approve the round(rate*N) LEAST risky applications (lowest PD); ties broken by order (stable)."""
    k = int(round(rate * len(s)))
    order = np.argsort(risk(pd.Series(s)).to_numpy(), kind="mergesort")      # least risky first
    a = np.zeros(len(s), bool); a[order[:k]] = True
    return a

def outcome(a, y):
    ag, ab = int(((y == 0) & a).sum()), int(((y == 1) & a).sum())
    rg, rb = int(((y == 0) & ~a).sum()), int(((y == 1) & ~a).sum())
    return {"approved": ag + ab, "approved_goods": ag, "approved_bads": ab,
            "approved_bad_rate": ab / (ag + ab) if ag + ab else np.nan, "rejected_goods": rg, "rejected_bads": rb}

rows, swap_rows = [], []
for r in APPROVAL_RATES:
    A = {m: approve_at_rate(df[f"score_{m}"].to_numpy(), r) for m in ("base", "chal")}
    O = {m: outcome(A[m], y) for m in A}
    for m in A:
        rows.append({"approval_rate": r, "model": NAMES[m], "cutoff_pd": float(df.loc[A[m], f"score_{m}"].max()), **O[m]})
    add_goods = O["chal"]["approved_goods"] - O["base"]["approved_goods"]
    bads_avoided = O["chal"]["rejected_bads"] - O["base"]["rejected_bads"]
    swap_in, swap_out = A["chal"] & ~A["base"], A["base"] & ~A["chal"]   # approved only by with-FTS / only by without-FTS
    swap_rows.append({
        "approval_rate": r, "approved_each": O["base"]["approved"],
        "bad_rate_without_fts": O["base"]["approved_bad_rate"], "bad_rate_with_fts": O["chal"]["approved_bad_rate"],
        "bad_rate_change_pp": 100 * (O["chal"]["approved_bad_rate"] - O["base"]["approved_bad_rate"]),
        "additional_goods_approved": add_goods, "additional_bads_rejected": bads_avoided,
        "additional_goods_per_1000": 1000 * add_goods / N, "additional_bads_rejected_per_1000": 1000 * bads_avoided / N,
        "relative_bad_reduction": bads_avoided / O["base"]["approved_bads"] if O["base"]["approved_bads"] else np.nan,
        "swap_in_n": int(swap_in.sum()), "swap_in_bad_rate": y[swap_in].mean() if swap_in.any() else np.nan,
        "swap_out_n": int(swap_out.sum()), "swap_out_bad_rate": y[swap_out].mean() if swap_out.any() else np.nan})
APPROVAL = pd.DataFrame(rows)
SWAP = pd.DataFrame(swap_rows)
# Same number approved by both models -> every extra good approved replaces a bad: the two differences are equal
assert (SWAP["additional_goods_approved"] == SWAP["additional_bads_rejected"]).all()

# Suggested operating point (analytical only; config.strategy.proposed_cutoff is not set):
# within a realistic operating range (50% is very tight, 95% is close to approve-all), the approval rate with the
# largest bads avoided per 1,000 applications; ties -> higher approval rate.
OPERATING_RANGE = (0.60, 0.90)
cand = SWAP[SWAP["approval_rate"].between(*OPERATING_RANGE)]
best = cand.sort_values(["additional_bads_rejected_per_1000", "approval_rate"], ascending=False).iloc[0]
RECOMMENDED_RATE = float(best["approval_rate"])
print("Per model at each approval rate"); display(APPROVAL)
print("\nDifference, with FTS minus without FTS"); display(SWAP)
print(f"\nSuggested operating point: {RECOMMENDED_RATE:.0%} approval -> {int(best['additional_bads_rejected'])} bads avoided "
      f"({best['additional_bads_rejected_per_1000']:.1f} per 1,000), approved bad rate "
      f"{best['bad_rate_without_fts']:.2%} -> {best['bad_rate_with_fts']:.2%}")

In [0]:
# NOTE: prob_C_cal of the two models comes from different LightGBM models with their own Platt calibration.
# The same PD value does not select the same share of applications, so this view mixes a change in
# approval volume with a change in risk ranking. Use the equal-approval-rate view (step 4) for the comparison.
raw_rows = []
for c in RAW_CUTOFFS:
    o = {}
    for m in ("base", "chal"):
        a = (df[f"score_{m}"] < c).to_numpy() if HIGHER_IS_RISKIER else (df[f"score_{m}"] > c).to_numpy()
        o[m] = outcome(a, y)
        raw_rows.append({"cutoff_pd": c, "model": NAMES[m], "approval_rate": o[m]["approved"] / N, **o[m]})
    raw_rows.append({"cutoff_pd": c, "model": "difference (with - without)",
                     "approval_rate": (o["chal"]["approved"] - o["base"]["approved"]) / N,
                     **{k: o["chal"][k] - o["base"][k] for k in o["base"]}})
RAW_VIEW = pd.DataFrame(raw_rows)
print("Same raw PD cut-off (approve if prob_C_cal < cut-off). Raw scores are NOT directly comparable between the two models.")
display(RAW_VIEW)

In [0]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
w = 0.4
for i, m in enumerate(NAMES.values()):
    d = DECILES[DECILES.model == m]
    ax[0].bar(d["decile"] + (i - 0.5) * w, 100 * d["bad_rate"], width=w, label=m)
ax[0].axhline(100 * NB / N, color="grey", ls="--", lw=1, label=f"overall {NB / N:.1%}")
ax[0].set(title="Bad rate by decile (1 = highest PD)", xlabel="decile", ylabel="bad rate (%)", xticks=range(1, 11))
ax[0].legend()

x = np.arange(len(SWAP))
ax[1].bar(x - w / 2, SWAP["additional_goods_per_1000"], width=w, label="additional goods approved")
ax[1].bar(x + w / 2, SWAP["additional_bads_rejected_per_1000"], width=w, label="additional bads rejected")
ax[1].set_xticks(x, [f"{r:.0%}" for r in SWAP["approval_rate"]])
ax[1].set(title="With FTS minus without FTS, per 1,000 applications", xlabel="approval rate", ylabel="per 1,000")
ax[1].axvline(list(SWAP["approval_rate"]).index(RECOMMENDED_RATE), color="grey", ls=":", lw=1)
ax[1].legend()
fig.tight_layout()
save_fig(fig, OUT_PREFIX + "charts.png")
display(fig)
plt.close(fig)

OUTPUTS = {"counts": COUNTS, "deciles": DECILES, "performance": PERF, "paired_bootstrap": PAIRED,
           "approval_rate_by_model": APPROVAL, "approval_rate_difference": SWAP, "raw_cutoff_view": RAW_VIEW}
for k, v in OUTPUTS.items():
    save_csv(v, f"{OUT_PREFIX}{k}.csv")

In [0]:
# ── Business assumptions for this analysis (given by the business; not model parameters, so not in config) ──
PROFIT_GOOD        = 2000     # AED profit per approved good customer per year
LOSS_BAD           = 5000     # AED loss per approved bad customer per year
FTS_COST_PER_MONTH = 30000    # AED paid for FTS data per month
VALUE_YEARS        = 1        # years of profit / loss counted per approved customer
# Every OOT application was already approved by the current policy, so the model is an extra decline filter on top of
# it: only high approval rates make sense (decline the riskiest 2.5%-20% of already-approved customers).
ECON_APPROVAL_RATES = [0.80, 0.85, 0.90, 0.925, 0.95, 0.975, 1.00]
BREAKEVEN_PD = PROFIT_GOOD / (PROFIT_GOOD + LOSS_BAD)    # approve iff expected value > 0  <=>  PD < 2000/7000

# OOT period length and FTS cost for that period (cost is per month, independent of volume)
o0, o1 = (pd.Timestamp(x) for x in cfg["windows"]["OOT"])
OOT_MONTHS = (o1.year - o0.year) * 12 + (o1.month - o0.month)          # 2025-01-01 .. 2025-07-01 -> 6
FTS_COST = FTS_COST_PER_MONTH * OOT_MONTHS
# Labelled (GOOD+BAD) rows are only part of the OOT applications. Scale to all applications that would be scored
# (all OOT rows except policy-EXCLUDED), assuming the unlabelled ones swap like the labelled ones.
all_oot = pd.read_csv(BASE_CSV, usecols=[STATUS])
N_SCORED = int((all_oot[STATUS] != "EXCLUDED").sum())
SCALE = N_SCORED / N
GB_VALUE = (PROFIT_GOOD + LOSS_BAD) * VALUE_YEARS                       # value of swapping one bad for one good

def profit(a, yv):
    return VALUE_YEARS * (PROFIT_GOOD * int(((yv == 0) & a).sum()) - LOSS_BAD * int(((yv == 1) & a).sum()))

by_model, diff_rows = [], []
for r in ECON_APPROVAL_RATES:
    A = {m: approve_at_rate(df[f"score_{m}"].to_numpy(), r) for m in ("base", "chal")}
    P = {m: profit(A[m], y) for m in A}
    for m in A:
        o = outcome(A[m], y)
        by_model.append({"approval_rate": r, "model": NAMES[m], "cutoff_pd": float(df.loc[A[m], f"score_{m}"].max()),
                         **o, "profit_labelled_aed": P[m], "profit_scaled_aed": P[m] * SCALE})
    inc = P["chal"] - P["base"]
    swaps = outcome(A["chal"], y)["approved_goods"] - outcome(A["base"], y)["approved_goods"]
    diff_rows.append({"approval_rate": r, "declined_share": 1 - r, "extra_goods_approved": swaps, "bads_avoided": swaps,
                      "incremental_profit_labelled_aed": inc, "incremental_profit_scaled_aed": inc * SCALE,
                      "fts_cost_aed": FTS_COST, "net_labelled_aed": inc - FTS_COST, "net_scaled_aed": inc * SCALE - FTS_COST,
                      "breakeven_swaps_scaled": FTS_COST / GB_VALUE, "breakeven_swaps_labelled": FTS_COST / GB_VALUE / SCALE,
                      "breakeven_fts_price_per_month_aed": inc * SCALE / OOT_MONTHS})
ECON_BY_MODEL = pd.DataFrame(by_model)
ECON_DIFF = pd.DataFrame(diff_rows)
assert (ECON_DIFF["incremental_profit_labelled_aed"] == GB_VALUE * ECON_DIFF["extra_goods_approved"]).all(), \
    "incremental profit must equal (good profit + bad loss) x swaps at equal approval"

# Uncertainty: stratified bootstrap of applications (config.selection.bootstrap n / seed), approval re-applied per resample
BOOT = cfg["selection"]["bootstrap"]
rng = np.random.default_rng(BOOT["seed"])
i_good, i_bad = np.flatnonzero(y == 0), np.flatnonzero(y == 1)
sb, sc = df["score_base"].to_numpy(), df["score_chal"].to_numpy()
boot = np.empty((BOOT["n"], len(ECON_APPROVAL_RATES)))
for b in range(BOOT["n"]):
    idx = np.concatenate([rng.choice(i_good, len(i_good)), rng.choice(i_bad, len(i_bad))])
    yb = y[idx]
    for k, r in enumerate(ECON_APPROVAL_RATES):
        boot[b, k] = profit(approve_at_rate(sc[idx], r), yb) - profit(approve_at_rate(sb[idx], r), yb)
ECON_DIFF["incr_scaled_ci_low_aed"] = np.percentile(boot, 2.5, axis=0) * SCALE
ECON_DIFF["incr_scaled_ci_high_aed"] = np.percentile(boot, 97.5, axis=0) * SCALE
ECON_DIFF["prob_net_scaled_positive"] = (boot * SCALE > FTS_COST).mean(axis=0)

# Profit-maximising rule with calibrated PDs: approve iff PD < BREAKEVEN_PD (each model at its own volume)
pd_rows = []
for m in ("base", "chal"):
    a = (df[f"score_{m}"] < BREAKEVEN_PD).to_numpy() if HIGHER_IS_RISKIER else (df[f"score_{m}"] > BREAKEVEN_PD).to_numpy()
    pd_rows.append({"rule": f"approve if PD < {BREAKEVEN_PD:.4f}", "model": NAMES[m], "approval_rate": a.mean(),
                    **outcome(a, y), "profit_labelled_aed": profit(a, y), "profit_scaled_aed": profit(a, y) * SCALE})
pd_rows.append({"rule": "approve all (current policy)", "model": "-", "approval_rate": 1.0, **outcome(np.ones(N, bool), y),
                "profit_labelled_aed": profit(np.ones(N, bool), y), "profit_scaled_aed": profit(np.ones(N, bool), y) * SCALE})
ECON_PD_RULE = pd.DataFrame(pd_rows)
pd_inc = (ECON_PD_RULE.loc[1, "profit_scaled_aed"] - ECON_PD_RULE.loc[0, "profit_scaled_aed"])

# Decision view: each model at its OWN most profitable tested approval rate (incl. 100% = approve all), per OOT period
# and annualised (12 / OOT_MONTHS cohorts per year; FTS cost x 12 months)
best_rows = []
for m in NAMES.values():
    em = ECON_BY_MODEL[ECON_BY_MODEL.model == m]
    i = em["profit_labelled_aed"].idxmax()
    best_rows.append({"model": m, "best_approval_rate": em.at[i, "approval_rate"], "cutoff_pd": em.at[i, "cutoff_pd"],
                      "profit_labelled_aed": em.at[i, "profit_labelled_aed"], "profit_scaled_aed": em.at[i, "profit_scaled_aed"]})
ECON_BEST = pd.DataFrame(best_rows)
best_inc = ECON_BEST.loc[1, "profit_scaled_aed"] - ECON_BEST.loc[0, "profit_scaled_aed"]
PER_YEAR = 12 / OOT_MONTHS
ECON_DECISION = pd.DataFrame([
    {"view": v, "incremental_profit_period_aed": inc_, "fts_cost_period_aed": FTS_COST, "net_period_aed": inc_ - FTS_COST,
     "net_per_year_aed": (inc_ - FTS_COST) * PER_YEAR}
    for v, inc_ in [("same approval rate 95%", float(ECON_DIFF.set_index("approval_rate").loc[0.95, "incremental_profit_scaled_aed"])),
                    (f"PD rule (approve if PD < {BREAKEVEN_PD:.2%})", pd_inc),
                    ("each model at its best tested approval rate", best_inc)]])

ECON_ASSUMPTIONS = pd.DataFrame([
    {"item": "profit per good customer per year (AED)", "value": PROFIT_GOOD},
    {"item": "loss per bad customer per year (AED)", "value": LOSS_BAD},
    {"item": "years of value per customer", "value": VALUE_YEARS},
    {"item": "FTS cost per month (AED)", "value": FTS_COST_PER_MONTH},
    {"item": "OOT months", "value": OOT_MONTHS}, {"item": "FTS cost for the OOT period (AED)", "value": FTS_COST},
    {"item": "labelled applications (GOOD+BAD)", "value": N}, {"item": "scored applications (OOT excl. EXCLUDED)", "value": N_SCORED},
    {"item": "scale factor scored / labelled", "value": SCALE},
    {"item": "value of one swap (bad -> good) (AED)", "value": GB_VALUE},
    {"item": "break-even PD (approve if PD below)", "value": BREAKEVEN_PD},
    {"item": "break-even swaps per OOT period (scaled)", "value": FTS_COST / GB_VALUE}])

print(f"FTS cost for the {OOT_MONTHS}-month OOT period: {FTS_COST:,.0f} AED | one bad->good swap is worth {GB_VALUE:,.0f} AED "
      f"-> break-even {FTS_COST / GB_VALUE:.1f} swaps on {N_SCORED:,} scored applications "
      f"({FTS_COST / GB_VALUE / SCALE:.1f} on the {N:,} labelled)")
display(ECON_ASSUMPTIONS)
print("\nProfit per model at high approval rates (labelled = GOOD+BAD only; scaled = x" f"{SCALE:.3f})")
display(ECON_BY_MODEL)
print("\nWith FTS minus without FTS, against the FTS cost")
display(ECON_DIFF)
print(f"\nProfit-maximising PD rule (approve if PD < {BREAKEVEN_PD:.2%}): incremental profit with FTS {pd_inc:,.0f} AED "
      f"(scaled) vs FTS cost {FTS_COST:,.0f} -> net {pd_inc - FTS_COST:,.0f} AED")
display(ECON_PD_RULE)
print("\nEach model at its own best approval rate"); display(ECON_BEST)
print(f"\nDecision summary (scaled to {N_SCORED:,} scored applications; per year = x{PER_YEAR:g})"); display(ECON_DECISION)

# Chart: incremental profit (scaled, 95% CI) vs FTS cost
fig, ax = plt.subplots(figsize=(9, 4.5))
xs = np.arange(len(ECON_DIFF))
mid = ECON_DIFF["incremental_profit_scaled_aed"] / 1000
err = [mid - ECON_DIFF["incr_scaled_ci_low_aed"] / 1000, ECON_DIFF["incr_scaled_ci_high_aed"] / 1000 - mid]
ax.bar(xs, mid, yerr=err, capsize=4, color="tab:blue", label="incremental profit with FTS (95% CI)")
ax.axhline(FTS_COST / 1000, color="tab:red", ls="--", label=f"FTS cost {OOT_MONTHS} months ({FTS_COST / 1000:,.0f}k)")
ax.axhline(0, color="grey", lw=0.8)
ax.set_xticks(xs, [f"{r:.1%}" for r in ECON_DIFF["approval_rate"]])
ax.set(title="FTS value per 6-month cohort vs cost (scaled to all scored OOT applications)",
       xlabel="approval rate among already-approved applications", ylabel="AED thousand")
ax.legend(loc="upper left")
fig.tight_layout()
save_fig(fig, OUT_PREFIX + "economics.png")
display(fig)
plt.close(fig)

ECON_OUT = {"econ_assumptions": ECON_ASSUMPTIONS, "econ_by_model": ECON_BY_MODEL, "econ_difference": ECON_DIFF,
            "econ_pd_rule": ECON_PD_RULE, "econ_best": ECON_BEST, "econ_decision": ECON_DECISION}
for k, v in ECON_OUT.items():
    save_csv(v, f"{OUT_PREFIX}{k}.csv")
OUTPUTS.update(ECON_OUT)

## Conclusion

On the same 4,509 OOT applications (394 bads, bad rate 8.74%), the Model C with FTS data ranks risk better than the model without it: **Gini 0.522 vs 0.489 (+0.032, 95% CI +0.003 to +0.061)** and KS 0.415 vs 0.385. The Gini gain is statistically positive, but only just.

In business terms the gain is small. At the same approval rate, both models approve the same number of applications, so every extra good approved replaces a bad. At the suggested operating point of **60% approval**, the FTS model approves **14 more goods and 14 fewer bads (about 3 per 1,000 applications)**. The approved bad rate falls from 3.88% to 3.36%, which is 13% fewer approved bads. At 70–90% approval the gain is between −1 and +7 bads (under 2 per 1,000), which is within the noise of 394 bads. About 300 applications swap between the two approved sets at 60%, and the ones the FTS model brings in are lower-risk (4.9% bad rate) than the ones it pushes out (9.5%).

The same-PD cut-off view is not a like-for-like comparison: the FTS model's calibrated PDs are lower, so at the same cut-off it approves 1–6 points more applications.

### Economics of buying FTS (step 7)

Assumptions: each good customer brings +2,000 AED a year and each bad customer costs −5,000 AED a year, counted over one year. FTS costs 30,000 AED a month, which is 180,000 AED for the 6-month OOT cohort. Results are scaled from the 4,509 labelled applications to the 7,986 that would be scored, assuming the unlabelled ones behave the same way. Each bad swapped for a good is worth 7,000 AED, so FTS breaks even at about **26 swaps per 6 months** (about 3 per 1,000 applications).

All OOT applications were already approved, so the model works as an extra decline filter and only high approval rates are relevant. The economically optimal cut-off is PD < 2,000 / 7,000 = 28.6%, which declines about 5% of the approved customers.

| View | Extra profit from FTS (6 months) | Net of FTS cost (6 months) | Net per year |
| --- | --- | --- | --- |
| Same approval rate, 95% | +211k AED | **+31k AED** | +62k AED |
| PD rule, approve if PD < 28.6% (about 95% approval) | +225k AED | **+45k AED** | +90k AED |
| Each model at its own best approval rate (97.5% for both) | +112k AED | **−68k AED** | −137k AED |
| 80%–92.5% or 97.5% approval | −12k to +99k AED | −81k to −192k AED | loss |

At the economically optimal cut-off (about 95% approval), FTS roughly pays for itself, with a small gain of about +30k to +45k AED per 6 months. The bootstrap gives only a 63% chance that the net is positive, and at every other cut-off tested FTS loses money. At 30,000 AED a month, FTS is therefore **close to break-even rather than clearly profitable**. It would pay off clearly at about 18,000–35,000 AED a month or less, depending on the cut-off. Without FTS, declining customers adds little profit over approving everyone, so most of the value of the decline filter comes from the FTS data.

Caveats: the result counts only one year of value per customer. Multi-year profit would raise the gain roughly in proportion. The data covers already-approved customers only, so any FTS value from approving applicants who are declined today is not measured. Results are scaled from the labelled sample.

**Key finding:** FTS data adds a modest, statistically positive improvement in ranking (about +3 Gini points). It reduces approved bads clearly only at tighter approval rates (around 60%), and the benefit is marginal at 70% approval and above. The operating point is a modelling suggestion; `config.strategy.proposed_cutoff` is not set, so the business cut-off still needs to be decided.

In [0]:
# Re-read the saved CSVs and check the key facts independently of the in-memory tables
chk = {k: pd.read_csv(os.path.join(REPORTS, f"{OUT_PREFIX}{k}.csv")) for k in OUTPUTS}
d = chk["deciles"]
assert TARGET_MATCH and STATUS_MATCH, "bad flag differs between files"
assert (d.groupby("model")["count"].sum() == N).all() and (d.groupby("model")["bads"].sum() == NB).all(), "deciles do not cover the population"
assert (d.groupby("model")["count"].agg(lambda c: c.max() - c.min()) <= 1).all(), "deciles are not equal-count"
for m in ("base", "chal"):          # direction: decile 1 (highest PD) must hold the highest scores
    dm = d[d.model == NAMES[m]].set_index("decile")
    assert dm.loc[1, "score_min"] >= dm.loc[2, "score_max"] - 1e-12, f"{NAMES[m]}: decile direction wrong"
    assert abs(roc_auc_score(y, df[f"score_{m}"]) - chk["performance"].set_index("model").loc[NAMES[m], "auc"]) < 1e-12
a = chk["approval_rate_by_model"]
assert (a.groupby("approval_rate")["approved"].nunique() == 1).all(), "models approve different volumes at the same rate"
assert ((a["approved_goods"] + a["approved_bads"] + a["rejected_goods"] + a["rejected_bads"]) == N).all()
assert ((a["approved"] - (a["approval_rate"] * N).round()).abs() <= 0).all()
# Economics: recompute profit from the decile-independent counts in the saved per-model table
eb = chk["econ_by_model"]
assert (eb["profit_labelled_aed"] == VALUE_YEARS * (PROFIT_GOOD * eb["approved_goods"] - LOSS_BAD * eb["approved_bads"])).all()
assert (eb[eb.approval_rate == 1.0]["profit_labelled_aed"] == VALUE_YEARS * (PROFIT_GOOD * (N - NB) - LOSS_BAD * NB)).all()
ed = chk["econ_difference"]
assert np.allclose(ed["net_scaled_aed"], ed["incremental_profit_labelled_aed"] * SCALE - FTS_COST)
assert FTS_COST == FTS_COST_PER_MONTH * 6, "OOT window is not 6 months"

run_summary = {"module": "S05c_comparison_fts", "run_time": datetime.now().isoformat(timespec="seconds"),
               "elapsed_s": round(time.time() - _t0, 1), "git_commit": GIT_COMMIT, "config_hash": CONFIG_HASH,
               "inputs": {"without_fts": BASE_CSV, "with_fts": CHAL_CSV}, "n": N, "bads": NB,
               "performance": PERF.round(6).to_dict("records"), "paired_bootstrap": PAIRED.round(6).to_dict("records"),
               "recommended_approval_rate": RECOMMENDED_RATE,
               "economics": {"assumptions": ECON_ASSUMPTIONS.to_dict("records"), "decision": ECON_DECISION.round(0).to_dict("records")},
               "at_recommended": SWAP[SWAP.approval_rate == RECOMMENDED_RATE].round(6).to_dict("records")[0],
               "outputs": [f"{OUT_PREFIX}{k}.csv" for k in OUTPUTS] + [OUT_PREFIX + "charts.png", OUT_PREFIX + "economics.png"], "pass_fail": "PASS"}
os.makedirs(os.path.join(REPORTS, "run_summaries"), exist_ok=True)
_js = os.path.join(_TMP, "s.json")
with open(_js, "w") as fh:
    json.dump(run_summary, fh, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
shutil.copyfile(_js, os.path.join(REPORTS, "run_summaries", "s05_comparison_fts.json"))
print("\u2705 All checks passed; run summary saved.")

## Plan: Compare existing A-Score vs Model C (with FTS)

Below is the proposed plan. Please review and confirm before I proceed.

### Goal
Replicate the comparison notebook (steps 1–7 above) but with **A-Score (production)** as the baseline against with  "Model C without FTS" and "Model C with FTS", and keep **Model C with FTS** and **Model C without FTS** as the challenger. also compare the 2 models C as above.

### Questions I need answered first

1. **Which A-Score column is the PD / risk score?** The function loads the full `cc_a_score_adhoc_data_file` table. I need to know the exact column name that holds the probability of default (or risk score) to use for ranking. Is it something like `A_SCORE`, `PD`, `prob_default`, or another name? Please share the column name (or I can print the columns first). answer: it is called FINAL_SCORE

2. **Score direction:** Is higher A-Score = higher risk (like a PD), or higher = lower risk (like a credit score mapped via PDO)? This determines sort order for deciles and approval-rate analysis. answer: I think FINAL_SCORE is higher, the less risky

3. **Is the A-Score already calibrated to a PD?** If yes, we can directly compare PD cut-offs. If it's a log-odds or points scale, the "same raw PD cut-off" view won't be meaningful and we'll rely only on the equal-approval-rate view. answer: i think so

4. **Join key:** The function merges on `CRNNO`. I assume this exists in our current `df` (the OOT DataFrame used above). If `df` doesn't have `CRNNO`, what is the mapping column? answer: yes, if duplicate, keep the first one.

5. **Approval rates / economics assumptions:** Should I use the same rates and AED assumptions as above (profit per good = 2,000 AED, loss per bad = 5,000 AED, FTS cost = 30,000 AED/month), or do different assumptions apply for comparing against the production A-Score? answer: yes, same assumption

6. **Which applications to include?** The current comparison uses the OOT labelled set (GOOD + BAD only, N = 4,509). Should the A-Score comparison use the same population, or a different subset? answer: use the population

### Proposed steps (once questions are answered)

| Step | Description |
|------|-------------|
| **0. Load & merge** | Call `load_a_score(df_target)` with our OOT DataFrame. Inspect columns, check coverage (how many CRNNOs matched, how many have a non-null score). |
| **1. Align populations** | Keep only rows that have both a valid A-Score AND a valid Model C score AND a known label (GOOD/BAD). Report any drop in sample size. |
| **2. Counts & base rates** | N, number of bads, bad rate — same as step 1 of the existing notebook. |
| **3. Discriminatory power** | AUC, Gini, KS for both A-Score and Model C. Paired bootstrap for the Gini difference and 95% CI. Decile analysis (bad rate by decile). |
| **4. Equal approval-rate comparison** | At the same approval rates (e.g. 60%, 70%, 80%, 85%, 90%, 95%), compare approved bads, rejected goods, bad-rate reduction, swap-set analysis. |
| **5. Same raw cut-off view** | Only if both scores are on a comparable PD scale; otherwise skip. |
| **6. Charts & CSV outputs** | Decile bar chart, swap chart, save all tables. |
| **7. Economics** | Incremental profit of switching from A-Score to Model C, net of any cost difference, with bootstrap confidence intervals. |
| **8. Conclusion** | Narrative summary of findings. |

### What I will NOT change
- The existing cells above remain untouched.
- This comparison will be appended as new cells below.

---
**Please answer the questions above (especially #1 and #2) so I can proceed.**

In [0]:
# A-Score (production) = FINAL_SCORE in cc_a_score_adhoc_data_file (one row per customer, key CRNNO).
# FINAL_SCORE is a POINTS scale (528-834), HIGHER = LOWER risk; it is not a PD, so it is ranked with the opposite sign
# and the same-raw-PD-cut-off view is not applicable to it. Card-level OOT rows of the same customer share its A-Score.
# If a CRNNO had several A-Score rows, the first one (earliest BASE month) is kept.
from pyspark.sql import functions as F
A_TABLE, A_SCORE_COL, A_KEY = "mis_automation.silver_credit_card.cc_a_score_adhoc_data_file", "FINAL_SCORE", "CRNNO"
A_HIGHER_IS_RISKIER = False
OUT_PREFIX_A = "05d_ascore_comparison_"

def _clean_key(s):
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

keys = pd.read_csv(BASE_CSV, usecols=[KEY, A_KEY, STATUS, "APPRECDDATE"], dtype={KEY: str, A_KEY: str})
keys[KEY], keys[A_KEY] = _clean_key(keys[KEY]), _clean_key(keys[A_KEY])
crn_list = keys[A_KEY].dropna().unique().tolist()
a_raw = (spark.table(A_TABLE)
         .select(F.col(A_KEY).cast("string").alias(A_KEY), "BASE", F.col(A_SCORE_COL).cast("double").alias("ascore"))
         .filter(F.col(A_KEY).isin(crn_list)).toPandas())
a_raw[A_KEY] = _clean_key(a_raw[A_KEY])
A_DUP = int(a_raw.duplicated(A_KEY).sum())
a_one = (a_raw[a_raw["ascore"].notna()].sort_values([A_KEY, "BASE"], kind="mergesort")
         .drop_duplicates(A_KEY, keep="first"))

dfa = (df.merge(keys[[KEY, A_KEY, "APPRECDDATE"]], on=KEY, how="left", validate="one_to_one")
         .merge(a_one, on=A_KEY, how="left", validate="many_to_one"))
assert len(dfa) == N, "join on CRNNO changed the number of labelled applications"
app = pd.to_datetime(dfa["APPRECDDATE"])
dfa["ascore_lag_months"] = (dfa["BASE"].dt.year - app.dt.year) * 12 + (dfa["BASE"].dt.month - app.dt.month)

DA = dfa[dfa["ascore"].notna()].reset_index(drop=True)          # analysis population for this section
NA, NBA = len(DA), int(DA["bad"].sum())
# Scored (non-EXCLUDED) OOT applications that have an A-Score, used to scale the economics
k_all = keys.merge(a_one[[A_KEY]], on=A_KEY, how="inner")
N_SCORED_A = int((k_all[STATUS] != "EXCLUDED").sum())

COUNTS_A = pd.DataFrame([
    {"step": "OOT rows (all statuses)", "n": len(keys)},
    {"step": "OOT rows with an A-Score", "n": len(k_all)},
    {"step": "A-Score table rows for OOT customers", "n": len(a_raw)},
    {"step": "duplicate CRNNO rows dropped (keep first)", "n": A_DUP},
    {"step": "GOOD+BAD (previous section)", "n": N}, {"step": "GOOD+BAD bads (previous section)", "n": NB},
    {"step": "GOOD+BAD without A-Score (dropped)", "n": N - NA},
    {"step": "bads without A-Score (dropped)", "n": NB - NBA},
    {"step": "GOOD+BAD with A-Score (analysis population)", "n": NA}, {"step": "bads", "n": NBA}, {"step": "goods", "n": NA - NBA},
    {"step": "scored OOT rows (non-EXCLUDED) with A-Score", "n": N_SCORED_A}])
display(COUNTS_A)
print(f"Bad rate: {NBA / NA:.2%} ({NBA:,} / {NA:,}); A-Score range {DA['ascore'].min():.0f}-{DA['ascore'].max():.0f}")
print("A-Score month (BASE) minus application month:")
display(DA["ascore_lag_months"].value_counts().sort_index().rename_axis("lag_months").reset_index(name="n"))

In [0]:
# Three scores on the same population DA; each is turned into a risk score (larger = riskier) with an explicit sign,
# then the helpers of steps 2-4 are reused unchanged (they assume larger = riskier).
assert HIGHER_IS_RISKIER, "helpers above are reused on risk-oriented scores"
MODELS_A = {"ascore": "A-Score (production)", "base": "Model C without FTS", "chal": "Model C with FTS"}
COL_A = {"ascore": "ascore", "base": "score_base", "chal": "score_chal"}
SIGN_A = {"ascore": 1.0 if A_HIGHER_IS_RISKIER else -1.0, "base": 1.0, "chal": 1.0}
PAIRS_A = [("ascore", "base"), ("ascore", "chal"), ("base", "chal")]      # (baseline, challenger)
PAIR_NAME = {p: f"{MODELS_A[p[1]]} - {MODELS_A[p[0]]}" for p in PAIRS_A}
ya = DA["bad"].to_numpy()
RISK_A = {m: SIGN_A[m] * DA[COL_A[m]].to_numpy(dtype=float) for m in MODELS_A}

def decile_table_signed(yv, m):
    t = decile_table(yv, RISK_A[m], MODELS_A[m])
    if SIGN_A[m] < 0:      # report min / max on the original scale
        t["score_min"], t["score_max"] = -t["score_max"], -t["score_min"]
    return t

DECILES_A = pd.concat([decile_table_signed(ya, m) for m in MODELS_A], ignore_index=True)
PERF_A = pd.DataFrame([{
    "model": MODELS_A[m], "n": NA, "bads": NBA, "auc": roc_auc_score(ya, RISK_A[m]), "gini": 2 * roc_auc_score(ya, RISK_A[m]) - 1,
    "ks": common.ks_stat(ya, RISK_A[m]),
    "top10_bad_capture": DECILES_A.query("model == @MODELS_A[@m] and decile == 1")["cum_bad_capture"].iloc[0],
    "top20_bad_capture": DECILES_A.query("model == @MODELS_A[@m] and decile == 2")["cum_bad_capture"].iloc[0]}
    for m in MODELS_A])
PAIRED_A = pd.concat([common.paired_bootstrap(ya, RISK_A[c], RISK_A[b], cfg).assign(pair=PAIR_NAME[(b, c)])
                      for b, c in PAIRS_A], ignore_index=True)
PAIRED_A = PAIRED_A[["pair"] + [c for c in PAIRED_A.columns if c != "pair"]]
SPEARMAN_A = DA[["ascore", "score_base", "score_chal"]].corr(method="spearman")

for m in MODELS_A.values():
    print(f"\nDecile table: {m} (decile 1 = highest risk)")
    display(DECILES_A[DECILES_A.model == m])
print(f"\nOverall performance on {NA:,} applications with an A-Score"); display(PERF_A)
print("\nPaired bootstrap (challenger minus baseline)"); display(PAIRED_A)
print("\nSpearman rank correlation (A-Score is reversed: expect negative with the PDs)"); display(SPEARMAN_A)

In [0]:
# Same approval rate for every score: approve the round(rate*N) least risky. A-Score has integer points with ties;
# ties at the cut-off are broken by file order (stable sort), as in step 4.
rows_a, swap_a = [], []
for r in APPROVAL_RATES:
    AA = {m: approve_at_rate(RISK_A[m], r) for m in MODELS_A}
    OA = {m: outcome(AA[m], ya) for m in AA}
    for m in AA:
        appr = DA.loc[AA[m], COL_A[m]]
        rows_a.append({"approval_rate": r, "model": MODELS_A[m],
                       "cutoff": float(appr.min() if SIGN_A[m] < 0 else appr.max()), **OA[m]})
    for b, c in PAIRS_A:
        s_in, s_out = AA[c] & ~AA[b], AA[b] & ~AA[c]
        avoided = OA[c]["rejected_bads"] - OA[b]["rejected_bads"]
        swap_a.append({
            "pair": PAIR_NAME[(b, c)], "approval_rate": r, "approved_each": OA[b]["approved"],
            "bad_rate_baseline": OA[b]["approved_bad_rate"], "bad_rate_challenger": OA[c]["approved_bad_rate"],
            "bad_rate_change_pp": 100 * (OA[c]["approved_bad_rate"] - OA[b]["approved_bad_rate"]),
            "additional_goods_approved": OA[c]["approved_goods"] - OA[b]["approved_goods"], "additional_bads_rejected": avoided,
            "additional_bads_rejected_per_1000": 1000 * avoided / NA,
            "relative_bad_reduction": avoided / OA[b]["approved_bads"] if OA[b]["approved_bads"] else np.nan,
            "swap_in_n": int(s_in.sum()), "swap_in_bad_rate": ya[s_in].mean() if s_in.any() else np.nan,
            "swap_out_n": int(s_out.sum()), "swap_out_bad_rate": ya[s_out].mean() if s_out.any() else np.nan})
APPROVAL_A = pd.DataFrame(rows_a)
SWAP_A = pd.DataFrame(swap_a)
assert (SWAP_A["additional_goods_approved"] == SWAP_A["additional_bads_rejected"]).all()
print("Per score at each approval rate (A-Score cutoff = lowest approved FINAL_SCORE; Model C cutoff = highest approved PD)")
display(APPROVAL_A)
print("\nChallenger minus baseline"); display(SWAP_A)
print("\nSame raw cut-off view: not produced for the A-Score (FINAL_SCORE is a points scale, not a PD). "
      "The Model C with/without FTS raw-PD view is in step 5.")

In [0]:
# Uncertainty of "net bads avoided" (= challenger rejected bads - baseline rejected bads at the same approval rate).
# Stratified bootstrap of applications (config.selection.bootstrap n / seed, as in step 7); in each resample every score
# re-applies its own cut-off (approve the round(rate*n) least risky, stable sort), and all scores use the SAME resample.
# Approval rates: steps 4 and 7 combined (100% = approve all is left out: nothing is declined there).
CI_RATES = sorted((set(APPROVAL_RATES) | set(ECON_APPROVAL_RATES)) - {1.0})
BOOT_CI = cfg["selection"]["bootstrap"]

def rejected_bads(risk_s, yv, ks):
    """Rejected bads at each k approved; same rule as approve_at_rate (k least risky approved, mergesort ties)."""
    cum = np.cumsum(yv[np.argsort(risk_s, kind="mergesort")])
    return int(yv.sum()) - cum[ks - 1]

def net_bads_ci(yv, risks, pairs, names, population):
    n = len(yv)
    ks = np.array([int(round(r * n)) for r in CI_RATES])
    point = {m: rejected_bads(risks[m], yv, ks) for m in risks}
    rng_ci = np.random.default_rng(BOOT_CI["seed"])
    i_g, i_b = np.flatnonzero(yv == 0), np.flatnonzero(yv == 1)
    boot_ = {p: np.empty((BOOT_CI["n"], len(ks))) for p in pairs}
    for b in range(BOOT_CI["n"]):
        idx = np.concatenate([rng_ci.choice(i_g, len(i_g)), rng_ci.choice(i_b, len(i_b))])
        rb = {m: rejected_bads(risks[m][idx], yv[idx], ks) for m in risks}
        for base_m, chal_m in pairs:
            boot_[(base_m, chal_m)][b] = rb[chal_m] - rb[base_m]
    rows_ = []
    for base_m, chal_m in pairs:
        bt = boot_[(base_m, chal_m)]
        for k_i, r in enumerate(CI_RATES):
            est = int(point[chal_m][k_i] - point[base_m][k_i])
            lo, hi = np.percentile(bt[:, k_i], [2.5, 97.5])
            rows_.append({"population": population, "pair": f"{names[chal_m]} - {names[base_m]}", "approval_rate": r,
                          "declined_each": n - ks[k_i], "baseline_rejected_bads": int(point[base_m][k_i]),
                          "challenger_rejected_bads": int(point[chal_m][k_i]), "net_bads_avoided": est,
                          "boot_mean": bt[:, k_i].mean(), "boot_se": bt[:, k_i].std(ddof=1), "ci_low": lo, "ci_high": hi,
                          "p_le_0": (bt[:, k_i] <= 0).mean(),
                          "significant": "yes, > 0" if lo > 0 else ("yes, < 0" if hi < 0 else "no (CI includes 0)")})
    return pd.DataFrame(rows_)

# (a) A-Score population (4,403): the three pairs of step 10
NET_CI_A = net_bads_ci(ya, RISK_A, PAIRS_A, MODELS_A, f"A-Score population (n={NA:,})")
# (b) Step-4 population (4,509): Model C with vs without FTS
NET_CI = net_bads_ci(y, {m: risk(df[f"score_{m}"]).to_numpy(dtype=float) for m in ("base", "chal")},
                     [("base", "chal")], {"base": "Model C without FTS", "chal": "Model C with FTS"},
                     f"Step-4 population (n={N:,})")

# Checks: point estimates must equal the swap-set tables of steps 4 and 10 at the shared approval rates
chk_a = NET_CI_A.merge(SWAP_A[["pair", "approval_rate", "additional_bads_rejected"]], on=["pair", "approval_rate"])
assert len(chk_a) == len(PAIRS_A) * len(set(CI_RATES) & set(APPROVAL_RATES)), "pair / rate mismatch with SWAP_A"
assert (chk_a["net_bads_avoided"] == chk_a["additional_bads_rejected"]).all(), "point estimate differs from step 10"
chk_4 = NET_CI.merge(SWAP[["approval_rate", "additional_bads_rejected"]], on="approval_rate")
assert (chk_4["net_bads_avoided"] == chk_4["additional_bads_rejected"]).all(), "point estimate differs from step 4"
chk_7 = NET_CI.merge(ECON_DIFF[["approval_rate", "bads_avoided"]], on="approval_rate")
assert (chk_7["net_bads_avoided"] == chk_7["bads_avoided"]).all(), "point estimate differs from step 7"
assert (NET_CI_A["ci_low"] <= NET_CI_A["ci_high"]).all() and (NET_CI["ci_low"] <= NET_CI["ci_high"]).all()

cols_show = ["pair", "approval_rate", "declined_each", "baseline_rejected_bads", "challenger_rejected_bads",
             "net_bads_avoided", "boot_se", "ci_low", "ci_high", "p_le_0", "significant"]
print(f"Net bads avoided (challenger - baseline), 95% stratified bootstrap CI, {BOOT_CI['n']:,} resamples")
print(f"\nA-Score population (n={NA:,}, {NBA} bads)"); display(NET_CI_A[cols_show])
print(f"\nStep-4 population (n={N:,}, {NB} bads)"); display(NET_CI[cols_show])

# Chart: point estimate with 95% CI per pair (A-Score population)
fig, axc = plt.subplots(figsize=(11, 4.5))
xr = np.arange(len(CI_RATES))
for i, p in enumerate(PAIRS_A):
    d = NET_CI_A[NET_CI_A.pair == PAIR_NAME[p]]
    axc.errorbar(xr + (i - 1) * 0.22, d["net_bads_avoided"],
                 yerr=[d["net_bads_avoided"] - d["ci_low"], d["ci_high"] - d["net_bads_avoided"]],
                 fmt="o", capsize=3, label=PAIR_NAME[p])
axc.axhline(0, color="grey", lw=0.8)
axc.set_xticks(xr, [f"{r:.1%}" for r in CI_RATES])
axc.set(title=f"Net bads avoided at equal approval rate, 95% bootstrap CI (n={NA:,})", xlabel="approval rate",
        ylabel="bads avoided")
axc.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, OUT_PREFIX_A + "net_bads_ci.png")
display(fig)
plt.close(fig)

save_csv(NET_CI_A, OUT_PREFIX_A + "net_bads_bootstrap_ci.csv")
save_csv(NET_CI, OUT_PREFIX + "net_bads_bootstrap_ci.csv")

In [0]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
w = 0.27
for i, m in enumerate(MODELS_A.values()):
    d = DECILES_A[DECILES_A.model == m]
    ax[0].bar(d["decile"] + (i - 1) * w, 100 * d["bad_rate"], width=w, label=m)
ax[0].axhline(100 * NBA / NA, color="grey", ls="--", lw=1, label=f"overall {NBA / NA:.1%}")
ax[0].set(title="Bad rate by decile (1 = highest risk)", xlabel="decile", ylabel="bad rate (%)", xticks=range(1, 11))
ax[0].legend()

x = np.arange(len(APPROVAL_RATES))
for i, p in enumerate(PAIRS_A):
    d = SWAP_A[SWAP_A.pair == PAIR_NAME[p]]
    ax[1].bar(x + (i - 1) * w, d["additional_bads_rejected_per_1000"], width=w, label=PAIR_NAME[p])
ax[1].axhline(0, color="grey", lw=0.8)
ax[1].set_xticks(x, [f"{r:.0%}" for r in APPROVAL_RATES])
ax[1].set(title="Bads avoided at equal approval rate, per 1,000 applications", xlabel="approval rate", ylabel="per 1,000")
ax[1].legend(fontsize=8)
fig.tight_layout()
save_fig(fig, OUT_PREFIX_A + "charts.png")
display(fig)
plt.close(fig)

OUTPUTS_A = {"counts": COUNTS_A, "deciles": DECILES_A, "performance": PERF_A, "paired_bootstrap": PAIRED_A,
             "spearman": SPEARMAN_A.reset_index(names="score"), "approval_rate_by_model": APPROVAL_A,
             "approval_rate_difference": SWAP_A}
for k, v in OUTPUTS_A.items():
    save_csv(v, f"{OUT_PREFIX_A}{k}.csv")

In [0]:
# Same business assumptions as step 7 (PROFIT_GOOD, LOSS_BAD, FTS_COST_PER_MONTH, VALUE_YEARS, ECON_APPROVAL_RATES).
# Data cost of a switch: Model C with FTS needs FTS data (FTS_COST per OOT period); A-Score and Model C without FTS
# add no data cost. Other costs (build, run, change) are not included.
COST_A = {"ascore": 0.0, "base": 0.0, "chal": float(FTS_COST)}
SCALE_A = N_SCORED_A / NA

by_a, diff_a = [], []
for r in ECON_APPROVAL_RATES:
    AA = {m: approve_at_rate(RISK_A[m], r) for m in MODELS_A}
    P = {m: profit(AA[m], ya) for m in AA}
    for m in AA:
        by_a.append({"approval_rate": r, "model": MODELS_A[m], **outcome(AA[m], ya),
                     "profit_labelled_aed": P[m], "profit_scaled_aed": P[m] * SCALE_A})
    for b, c in PAIRS_A:
        inc, cost = P[c] - P[b], COST_A[c] - COST_A[b]
        swaps = outcome(AA[c], ya)["approved_goods"] - outcome(AA[b], ya)["approved_goods"]
        diff_a.append({"pair": PAIR_NAME[(b, c)], "approval_rate": r, "swaps": swaps,
                       "incremental_profit_labelled_aed": inc, "incremental_profit_scaled_aed": inc * SCALE_A,
                       "extra_data_cost_aed": cost, "net_scaled_aed": inc * SCALE_A - cost})
ECON_BY_MODEL_A = pd.DataFrame(by_a)
ECON_DIFF_A = pd.DataFrame(diff_a)
assert (ECON_DIFF_A["incremental_profit_labelled_aed"] == GB_VALUE * ECON_DIFF_A["swaps"]).all()

# Stratified bootstrap (config.selection.bootstrap), approval re-applied per resample, same resamples for all pairs
rng_a = np.random.default_rng(BOOT["seed"])
ig, ib = np.flatnonzero(ya == 0), np.flatnonzero(ya == 1)
boot_a = np.empty((BOOT["n"], len(PAIRS_A), len(ECON_APPROVAL_RATES)))
for bi in range(BOOT["n"]):
    idx = np.concatenate([rng_a.choice(ig, len(ig)), rng_a.choice(ib, len(ib))])
    yb = ya[idx]
    for k, r in enumerate(ECON_APPROVAL_RATES):
        Pb = {m: profit(approve_at_rate(RISK_A[m][idx], r), yb) for m in MODELS_A}
        for j, (b, c) in enumerate(PAIRS_A):
            boot_a[bi, j, k] = Pb[c] - Pb[b]
ci = {"incr_scaled_ci_low_aed": [], "incr_scaled_ci_high_aed": [], "prob_net_positive": []}
for _, row in ECON_DIFF_A.iterrows():
    j = [PAIR_NAME[p] for p in PAIRS_A].index(row["pair"]); k = ECON_APPROVAL_RATES.index(row["approval_rate"])
    v = boot_a[:, j, k] * SCALE_A
    ci["incr_scaled_ci_low_aed"].append(np.percentile(v, 2.5)); ci["incr_scaled_ci_high_aed"].append(np.percentile(v, 97.5))
    ci["prob_net_positive"].append(float((v - row["extra_data_cost_aed"] > 0).mean()))
for k_, v_ in ci.items():
    ECON_DIFF_A[k_] = v_

# Decision view per pair: same approval rate 95%, and each score at its own most profitable tested approval rate
best_a = []
for m in MODELS_A.values():
    em = ECON_BY_MODEL_A[ECON_BY_MODEL_A.model == m]
    i = em["profit_labelled_aed"].idxmax()
    best_a.append({"model": m, "best_approval_rate": em.at[i, "approval_rate"], "profit_scaled_aed": em.at[i, "profit_scaled_aed"]})
ECON_BEST_A = pd.DataFrame(best_a).set_index("model")
dec = []
for b, c in PAIRS_A:
    cost = COST_A[c] - COST_A[b]
    sel = (ECON_DIFF_A["pair"] == PAIR_NAME[(b, c)]) & (ECON_DIFF_A["approval_rate"] == 0.95)
    at95 = float(ECON_DIFF_A.loc[sel, "incremental_profit_scaled_aed"].iloc[0])
    bvb = ECON_BEST_A.at[MODELS_A[c], "profit_scaled_aed"] - ECON_BEST_A.at[MODELS_A[b], "profit_scaled_aed"]
    for view, inc in (("same approval rate 95%", at95), ("each score at its best tested approval rate", bvb)):
        dec.append({"pair": PAIR_NAME[(b, c)], "view": view, "incremental_profit_period_aed": inc,
                    "extra_data_cost_period_aed": cost, "net_period_aed": inc - cost, "net_per_year_aed": (inc - cost) * PER_YEAR})
ECON_DECISION_A = pd.DataFrame(dec)

print(f"Scaled from {NA:,} labelled to {N_SCORED_A:,} scored applications with an A-Score (x{SCALE_A:.3f}); "
      f"FTS cost {FTS_COST:,.0f} AED per {OOT_MONTHS}-month OOT period")
print("\nProfit per score"); display(ECON_BY_MODEL_A)
print("\nChallenger minus baseline"); display(ECON_DIFF_A)
print("\nEach score at its own best tested approval rate"); display(ECON_BEST_A.reset_index())
print("\nDecision summary"); display(ECON_DECISION_A)

fig, ax = plt.subplots(figsize=(10, 4.5))
xs = np.arange(len(ECON_APPROVAL_RATES))
for i, p in enumerate(PAIRS_A):
    d = ECON_DIFF_A[ECON_DIFF_A.pair == PAIR_NAME[p]]
    mid = d["incremental_profit_scaled_aed"].to_numpy() / 1000
    err = [mid - d["incr_scaled_ci_low_aed"] / 1000, d["incr_scaled_ci_high_aed"] / 1000 - mid]
    ax.bar(xs + (i - 1) * 0.27, mid, width=0.27, yerr=err, capsize=3, label=PAIR_NAME[p])
ax.axhline(FTS_COST / 1000, color="tab:red", ls="--", label=f"FTS cost {OOT_MONTHS} months ({FTS_COST / 1000:,.0f}k)")
ax.axhline(0, color="grey", lw=0.8)
ax.set_xticks(xs, [f"{r:.1%}" for r in ECON_APPROVAL_RATES])
ax.set(title="Incremental profit per 6-month cohort (scaled, 95% CI)", xlabel="approval rate among already-approved", ylabel="AED thousand")
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
save_fig(fig, OUT_PREFIX_A + "economics.png")
display(fig)
plt.close(fig)

ECON_OUT_A = {"econ_by_model": ECON_BY_MODEL_A, "econ_difference": ECON_DIFF_A,
              "econ_best": ECON_BEST_A.reset_index(), "econ_decision": ECON_DECISION_A}
for k, v in ECON_OUT_A.items():
    save_csv(v, f"{OUT_PREFIX_A}{k}.csv")
OUTPUTS_A.update(ECON_OUT_A)

## Conclusion: production A-Score vs Model C (with and without FTS)

**Population**: 4,403 of the 4,509 labelled OOT applications have an A-Score (`FINAL_SCORE`, joined on `CRNNO`; 106 dropped, 1 of them bad), 393 bads (8.93%). The A-Score table has one row per customer (no duplicates to drop); 80% of scores are from the application month and 19% from the next month. `FINAL_SCORE` is a points scale (558–764 here, higher = lower risk, confirmed by the data), not a PD, so only the equal-approval-rate view is used for it.

| Score | Gini | KS | Bads in riskiest 10% |
| --- | --- | --- | --- |
| A-Score (production) | 0.369 | 0.285 | 27.2% |
| Model C without FTS | 0.484 | 0.381 | 29.5% |
| Model C with FTS | 0.518 | 0.410 | 30.3% |

Both Model C versions rank risk clearly better than the A-Score (paired bootstrap mean and 95% CI, 2,000 stratified resamples): **dGini +0.114 (95% CI +0.054 to +0.172) without FTS and +0.149 (+0.086 to +0.209) with FTS**. On this population FTS adds +0.035 (+0.005 to +0.064), consistent with the main comparison above. Rank correlation with the A-Score is only 0.47–0.55, so the models rank customers quite differently.

At equal approval rates, Model C avoids many more bads than the A-Score at tight cut-offs (60% approval: 36 fewer bads without FTS, 50 with FTS, about 8–11 per 1,000), but the gain shrinks at the high approval rates that matter here (85–97.5%: −5 to +14 bads, at most about 3 per 1,000).

**Economics** (same assumptions as step 7, scaled to 7,623 scored applications, 6-month OOT):

* Switching A-Score to Model C without FTS has no data cost: +61k AED per 6 months with each score at its best rate (97.5%), but −61k at 95% approval. The CI includes zero at every rate.
* Switching A-Score to Model C with FTS gives +146k to +170k AED at 85–97.5%, which is just below the 180k FTS cost (net −10k to −35k; probability of net gain 15–42%).
* FTS on top of Model C without FTS: net +50k at 95%, −83k at the best rate, so about break-even, as in step 7.

**Caveat**: every OOT application was approved by the current policy, which already uses the A-Score. The sample is truncated on the A-Score (its low-score applicants are missing), so the A-Score's measured power here is biased downwards relative to a through-the-door population, and the comparison favours Model C. The A-Score ranks with integer ties; ties at the cut-off are broken by file order.

In [0]:
# Re-read the saved CSVs and check key facts with methods independent of the tables built above
chk_a = {k: pd.read_csv(os.path.join(REPORTS, f"{OUT_PREFIX_A}{k}.csv")) for k in OUTPUTS_A}
d = chk_a["deciles"]
assert (d.groupby("model")["count"].sum() == NA).all() and (d.groupby("model")["bads"].sum() == NBA).all(), "deciles do not cover the population"
assert (d.groupby("model")["count"].agg(lambda c: c.max() - c.min()) <= 1).all(), "deciles are not equal-count"
dA = d[d.model == MODELS_A["ascore"]].set_index("decile")
assert dA.loc[1, "score_max"] <= dA.loc[2, "score_min"], "A-Score decile 1 must hold the LOWEST FINAL_SCORE"
assert dA.loc[1, "bad_rate"] > dA.loc[10, "bad_rate"], "A-Score direction: lowest scores must be riskiest"
# AUC of the raw FINAL_SCORE (no sign flip) must be 1 - reported AUC
p = chk_a["performance"].set_index("model")
assert abs((1 - roc_auc_score(ya, DA["ascore"])) - p.loc[MODELS_A["ascore"], "auc"]) < 1e-12
for m in ("base", "chal"):
    assert abs(roc_auc_score(ya, DA[COL_A[m]]) - p.loc[MODELS_A[m], "auc"]) < 1e-12
# Population: recount from the raw table in Spark (independent of the pandas merge)
a_cnt = spark.table(A_TABLE).select(F.col(A_KEY).cast("string").alias(A_KEY)).distinct().toPandas()
assert int(dfa[A_KEY].isin(set(a_cnt[A_KEY])).sum()) == NA, "A-Score coverage recount differs"
assert DA[KEY].is_unique and NA <= N
a = chk_a["approval_rate_by_model"]
assert (a.groupby("approval_rate")["approved"].nunique() == 1).all(), "scores approve different volumes at the same rate"
assert ((a["approved_goods"] + a["approved_bads"] + a["rejected_goods"] + a["rejected_bads"]) == NA).all()
eb = chk_a["econ_by_model"]
assert (eb["profit_labelled_aed"] == VALUE_YEARS * (PROFIT_GOOD * eb["approved_goods"] - LOSS_BAD * eb["approved_bads"])).all()
ed = chk_a["econ_difference"]
assert np.allclose(ed["net_scaled_aed"], ed["incremental_profit_labelled_aed"] * SCALE_A - ed["extra_data_cost_aed"])
assert (ed.loc[ed.pair.str.startswith("Model C without FTS"), "extra_data_cost_aed"] == 0).all()

run_summary_a = {"module": "S05d_comparison_ascore", "run_time": datetime.now().isoformat(timespec="seconds"),
                 "git_commit": GIT_COMMIT, "config_hash": CONFIG_HASH,
                 "inputs": {"a_score_table": A_TABLE, "a_score_column": A_SCORE_COL, "without_fts": BASE_CSV, "with_fts": CHAL_CSV},
                 "n": NA, "bads": NBA, "dropped_without_a_score": N - NA,
                 "performance": PERF_A.round(6).to_dict("records"), "paired_bootstrap": PAIRED_A.round(6).to_dict("records"),
                 "economics_decision": ECON_DECISION_A.round(0).to_dict("records"),
                 "outputs": [f"{OUT_PREFIX_A}{k}.csv" for k in OUTPUTS_A] + [OUT_PREFIX_A + "charts.png", OUT_PREFIX_A + "economics.png"],
                 "pass_fail": "PASS"}
_js = os.path.join(_TMP, "sa.json")
with open(_js, "w") as fh:
    json.dump(run_summary_a, fh, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
shutil.copyfile(_js, os.path.join(REPORTS, "run_summaries", "s05_comparison_ascore.json"))
print("\u2705 All A-Score comparison checks passed; run summary saved.")